# 06 — Run Ablations
Reads `config/ablations.yaml`, assembles features per run (tab/text/clip/roi), trains
Huber GBRT (log10) + isotonic calibration, and writes results to:
- `artifacts/results_ablations.csv`
- Per-run JSON: `artifacts/abl_<name>.json`

In [ ]:
import os, json, joblib, numpy as np, pandas as pd
from pathlib import Path

ART_DIR   = Path(os.getenv("ART_DIR",   "~/boxoffice/artifacts")).expanduser()
DATA_DIR  = Path(os.getenv("DATA_DIR",  "~/boxoffice/data")).expanduser()
IMG_DIR   = Path(os.getenv("IMG_DIR",   "~/boxoffice/data/img")).expanduser()

TEXT_DIR  = Path(os.getenv("TEXT_DIR",  ART_DIR / "text_emb")).expanduser()
CLIP_DIR  = Path(os.getenv("CLIP_DIR",  ART_DIR / "clip_emb")).expanduser()
ROI_DIR   = Path(os.getenv("ROI_DIR",   ART_DIR / "roi_feat")).expanduser()

def load_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))

# Handy helper
def _vec(path, dim):
    try:
        v = np.load(path); v = np.asarray(v, np.float32).ravel()
        if   v.size == dim: return v
        elif v.size >  dim: return v[:dim]
        else:               return np.pad(v, (0, dim - v.size))
    except Exception:
        return np.zeros(dim, np.float32)

In [ ]:
results_ablations = pd.read_csv(ART_DIR / "results_ablations.csv")
try:
    ablations_menu = load_json(ART_DIR / "ablations_menu.json")
except Exception:
    ablations_menu = None
# (Optional) per-run metadata: for p in ART_DIR.glob("abl_*.json"): ...

In [17]:
## FOR RELOAD ONLY
results_ablations = pd.read_csv(ART_DIR / "results_ablations.csv")
try:
    ablations_menu = load_json(ART_DIR / "ablations_menu.json")
except Exception:
    ablations_menu = None

In [1]:
import os as _os
_os.environ["TQDM_NOTEBOOK"] = "0"
from tqdm import tqdm
try: tqdm.monitor_interval = 0
except Exception: pass

In [5]:
# Imports & paths
import os, json, yaml
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

CONFIG     = Path(os.getenv("CONFIG_DIR", os.path.expanduser("~/boxoffice/config"))).expanduser()
ART_DIR    = Path(os.getenv("ART_DIR",    os.path.expanduser("~/boxoffice/artifacts"))).expanduser()
TEXT_DIR   = Path(os.getenv("TEXT_DIR",   str(ART_DIR / "text_emb"))).expanduser()
CLIP_DIR   = Path(os.getenv("CLIP_DIR",   str(ART_DIR / "clip_emb"))).expanduser()
ROI_DIR    = Path(os.getenv("ROI_DIR",    str(ART_DIR / "roi_feat"))).expanduser()
MASTER_CSV = Path(os.getenv("MASTER_CSV", os.path.expanduser("~/boxoffice/data/movies_master_preprocessed.csv"))).expanduser()

SEED = int(os.getenv("SEED","42"))

print("MASTER_CSV:", MASTER_CSV)
print("CONFIG   :", CONFIG)
print("ART_DIR  :", ART_DIR)
print("TEXT_DIR :", TEXT_DIR)
print("CLIP_DIR :", CLIP_DIR)
print("ROI_DIR  :", ROI_DIR)

MASTER_CSV: C:\Users\Vaishob\boxoffice\data\movies_master_preprocessed.csv
CONFIG   : C:\Users\Vaishob\boxoffice\config
ART_DIR  : C:\Users\Vaishob\boxoffice\artifacts
TEXT_DIR : C:\Users\Vaishob\boxoffice\artifacts\text_emb
CLIP_DIR : C:\Users\Vaishob\boxoffice\artifacts\clip_emb
ROI_DIR  : C:\Users\Vaishob\boxoffice\artifacts\roi_feat


In [6]:
# Load ablations config
abl_path = CONFIG / "ablations.yaml"
with open(abl_path, "r", encoding="utf-8") as f:
    ABL = yaml.safe_load(f)
runs = ABL.get("runs", [])
print(f"Loaded {len(runs)} runs")
display(pd.DataFrame([{"name": r.get("name"), **r.get("features", {})} for r in runs]))

Loaded 14 runs


,name,use_tabular,use_text,use_clip,use_roi
0,tab_only,True,False,False,False
1,text_only,False,True,False,False
2,clip_only,False,False,True,False
3,roi_only,False,False,False,True
4,tab_text,True,True,False,False
5,tab_clip,True,False,True,False
6,tab_roi,True,False,False,True
7,text_clip,False,True,True,False
8,text_roi,False,True,False,True
9,clip_roi,False,False,True,True


In [7]:
# Load data and splits; de-dup split IDs
df = pd.read_csv(MASTER_CSV)
if "tmdb_id" not in df.columns or "revenue_usd_2024" not in df.columns:
    raise ValueError("master must have tmdb_id and revenue_usd_2024")

def read_ids(name):
    p = ART_DIR / f"{name}.csv"
    if not p.exists():
        return []
    s = pd.read_csv(p)["tmdb_id"].dropna().astype("Int64").astype(int).tolist()
    # de-dup preserve order
    seen=set(); out=[]
    for v in s:
        if v not in seen:
            seen.add(v); out.append(int(v))
    return out

ids_train = read_ids("train_ids")
ids_val   = read_ids("val_ids")
ids_covid = read_ids("covid_diag_ids")
ids_test  = read_ids("test_ids")
print("split sizes:", dict(train=len(ids_train), val=len(ids_val), covid=len(ids_covid), test=len(ids_test)))

split sizes: {'train': 3215, 'val': 166, 'covid': 237, 'test': 465}


In [8]:
# De-dup dataframe by tmdb_id to get a unique index
dup_counts = df["tmdb_id"].value_counts()
dups = dup_counts[dup_counts>1]
print("Duplicate tmdb_id rows:", int(dups.shape[0]))
if not dups.empty:
    print("Top dup examples:", dups.head(10).to_dict())

# Keep-first policy (can be swapped if needed)
df_u = df.drop_duplicates(subset=["tmdb_id"], keep="first").copy()
df_idx = df_u.set_index("tmdb_id")

Duplicate tmdb_id rows: 310
Top dup examples: {18221.0: 6, 672.0: 3, 10200.0: 2, 36123.0: 2, 8055.0: 2, 46705.0: 2, 11499.0: 2, 1487.0: 2, 14359.0: 2, 11321.0: 2}


In [9]:
# Build global vocabularies from deduped df
NUM_COLS = [
    "budget_usd_2024","runtime","popularity","vote_average","vote_count",
    "wiki_pv_61d_log1p","reddit_posts_log1p","reddit_comments_log1p","gt_raw_sum"
]

TOPK_GENRES = 15
if "genres_norm" in df_u.columns:
    gcounts = {}
    for s in df_u["genres_norm"].fillna("").astype(str):
        for g in [x for x in s.split(";") if x]:
            gcounts[g] = gcounts.get(g,0) + 1
    GENRES = [g for g,_ in sorted(gcounts.items(), key=lambda kv: kv[1], reverse=True)[:TOPK_GENRES]]
else:
    GENRES = []

if "distributor_bucket" in df_u.columns:
    DIST_UNIQ = sorted(df_u["distributor_bucket"].fillna("other").astype(str).unique().tolist())
else:
    DIST_UNIQ = []

def build_tabular_strict(sub: pd.DataFrame) -> np.ndarray:
    n = len(sub)
    # numeric
    num_blocks = []
    for c in NUM_COLS:
        col = sub[c] if c in sub.columns else pd.Series(np.nan, index=sub.index)
        num_blocks.append(col.astype(float).fillna(0.0).to_numpy().reshape(n,1))
    Xnum = np.hstack(num_blocks) if num_blocks else np.zeros((n,0), dtype=np.float32)
    # genres
    if GENRES and "genres_norm" in sub.columns:
        G = np.zeros((n, len(GENRES)), dtype=np.float32)
        for i, s in enumerate(sub["genres_norm"].fillna("").astype(str)):
            toks = set([x for x in s.split(";") if x])
            for j,g in enumerate(GENRES):
                if g in toks: G[i,j]=1.0
        Xg = G
    else:
        Xg = np.zeros((n,0), dtype=np.float32)
    # distributor
    if DIST_UNIQ and "distributor_bucket" in sub.columns:
        cats = sub["distributor_bucket"].fillna("other").astype(str)
        map_idx = {u:i for i,u in enumerate(DIST_UNIQ)}
        Xd = np.zeros((n, len(DIST_UNIQ)), dtype=np.float32)
        for i,s in enumerate(cats):
            j = map_idx.get(s, None)
            if j is not None: Xd[i,j] = 1.0
    else:
        Xd = np.zeros((n,0), dtype=np.float32)

    X = np.hstack([Xnum, Xg, Xd]).astype(np.float32, copy=False)
    if X.shape[0] != n:
        raise ValueError(f"build_tabular_strict produced wrong row count: {X.shape} vs n={n}")
    return X

In [10]:
# Strict vector loaders (pad/truncate/zero-fill; preserve order)
from pathlib import Path
import numpy as np

def _as_1d(x):
    x = np.asarray(x)
    if x.ndim == 1: return x
    if x.ndim == 2 and 1 in x.shape: return x.reshape(-1)
    raise ValueError(f"Expected 1-D vector per id, got shape {x.shape}")

def load_vecs_strict(dir_path: Path, ids, name="feat"):
    dir_path = Path(dir_path)
    d = None; n_missing = n_malformed = n_fixdim = 0
    for tid in ids:
        fp = dir_path / f"{tid}.npy"
        if fp.exists():
            try:
                v = _as_1d(np.load(fp, allow_pickle=False))
                d = int(v.shape[0]); break
            except Exception:
                continue
    if d is None:
        return np.zeros((len(ids),0), dtype=np.float32), 0, len(ids), 0, 0
    rows = []
    for tid in ids:
        fp = dir_path / f"{tid}.npy"
        if not fp.exists():
            n_missing += 1; rows.append(np.zeros((d,), dtype=np.float32)); continue
        try:
            v = _as_1d(np.load(fp, allow_pickle=False)).astype(np.float32, copy=False)
        except Exception:
            n_malformed += 1; rows.append(np.zeros((d,), dtype=np.float32)); continue
        if v.shape[0] == d:
            rows.append(v)
        elif v.shape[0] > d:
            rows.append(v[:d]); n_fixdim += 1
        else:
            pad = np.zeros((d,), dtype=np.float32)
            pad[:v.shape[0]] = v
            rows.append(pad); n_fixdim += 1
    X = np.vstack(rows).astype(np.float32, copy=False)
    print(f"[{name}] shape={X.shape}, d={d}, missing={n_missing}, malformed={n_malformed}, fixed_dim={n_fixdim}")
    return X, d, n_missing, n_malformed, n_fixdim

In [11]:
# Assembly for a given toggle config
def assemble(ids, use_tab, use_text, use_clip, use_roi):
    ids = [int(i) for i in ids if int(i) in df_idx.index]
    sub = df_idx.loc[ids].copy()
    blocks=[]; names=[]
    if use_tab:
        X_tab = build_tabular_strict(sub); blocks.append(X_tab); names.append("tabular"); print(f"[tab ] shape={X_tab.shape}")
    if use_text:
        Xt,_,_,_,_ = load_vecs_strict(TEXT_DIR, ids, "text"); blocks.append(Xt); names.append("text")
    if use_clip:
        Xc,_,_,_,_ = load_vecs_strict(CLIP_DIR, ids, "clip"); blocks.append(Xc); names.append("clip")
    if use_roi:
        Xr,_,_,_,_ = load_vecs_strict(ROI_DIR,  ids, "roi");  blocks.append(Xr); names.append("roi")
    n=len(ids)
    for nm,Xb in zip(names,blocks):
        if Xb.shape[0] != n:
            raise ValueError(f"Row mismatch in '{nm}': {Xb.shape[0]} vs {n}")
    X = np.hstack(blocks) if blocks else np.zeros((n,0), dtype=np.float32)
    y = np.log10(np.maximum(sub["revenue_usd_2024"].astype(float).values, 1.0))
    return np.array(ids,dtype=int), X, y

In [12]:
# Train/eval helper
def smape(y_true_lin, y_pred_lin, eps=1e-9):
    den = (np.abs(y_true_lin) + np.abs(y_pred_lin)) + eps
    return 200.0 * float(np.mean(np.abs(y_pred_lin - y_true_lin) / den))

def compute_metrics(y_log, p_log):
    y = np.power(10.0, y_log); p = np.power(10.0, p_log)
    from sklearn.metrics import mean_absolute_error, r2_score
    return {
        "MAE_lin": float(mean_absolute_error(y, p)),
        "SMAPE": smape(y, p),
        "R2_log10": float(r2_score(y_log, p_log)),
        "MAE_log10": float(mean_absolute_error(y_log, p_log)),
    }

GB_PARAMS = dict(n_estimators=2000, learning_rate=0.02, max_depth=3, min_samples_leaf=20,
                 subsample=1.0, random_state=SEED, loss="huber", alpha=0.9)

In [13]:
# Run all ablations
rows = []
for run in runs:
    name = run.get("name")
    fcfg = run.get("features", {})
    use_tab  = bool(fcfg.get("use_tabular", True))
    use_text = bool(fcfg.get("use_text", False))
    use_clip = bool(fcfg.get("use_clip", False))
    use_roi  = bool(fcfg.get("use_roi",  False))
    print("\n=== Run:", name, fcfg, "===")
    # assemble
    tr_ids, Xtr, ytr = assemble(ids_train, use_tab, use_text, use_clip, use_roi)
    va_ids, Xva, yva = assemble(ids_val,   use_tab, use_text, use_clip, use_roi)
    te_ids, Xte, yte = assemble(ids_test,  use_tab, use_text, use_clip, use_roi)
    # train
    gb = GradientBoostingRegressor(**GB_PARAMS).fit(Xtr, ytr)
    # raw preds
    p_tr = gb.predict(Xtr); p_va = gb.predict(Xva); p_te = gb.predict(Xte)
    # isotonic on val
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, yva)
    p_tr_c = iso.transform(p_tr); p_va_c = iso.transform(p_va); p_te_c = iso.transform(p_te)
    # metrics
    for split, yt, pt in [("train", ytr, p_tr_c), ("val", yva, p_va_c), ("test", yte, p_te_c)]:
        m = compute_metrics(yt, pt); m.update({"run": name, "split": split,
                                               "use_tab": use_tab, "use_text": use_text,
                                               "use_clip": use_clip, "use_roi": use_roi})
        rows.append(m)
    # save per-run meta
    ART_DIR.mkdir(parents=True, exist_ok=True)
    outj = ART_DIR / f"abl_{name}.json"
    with open(outj, "w", encoding="utf-8") as f:
        json.dump({"name": name, "features": fcfg,
                   "shapes": {"Xtr": Xtr.shape, "Xva": Xva.shape, "Xte": Xte.shape}}, f, indent=2)
    print("Saved:", outj)


=== Run: tab_only {'use_tabular': True, 'use_text': False, 'use_clip': False, 'use_roi': False} ===
[tab ] shape=(3215, 55)
[tab ] shape=(166, 55)
[tab ] shape=(465, 55)
Saved: C:\Users\Vaishob\boxoffice\artifacts\abl_tab_only.json

=== Run: text_only {'use_tabular': False, 'use_text': True, 'use_clip': False, 'use_roi': False} ===
[text] shape=(3215, 768), d=768, missing=14, malformed=0, fixed_dim=0
[text] shape=(166, 768), d=768, missing=1, malformed=0, fixed_dim=0
[text] shape=(465, 768), d=768, missing=0, malformed=0, fixed_dim=0
Saved: C:\Users\Vaishob\boxoffice\artifacts\abl_text_only.json

=== Run: clip_only {'use_tabular': False, 'use_text': False, 'use_clip': True, 'use_roi': False} ===
[clip] shape=(3215, 768), d=768, missing=276, malformed=0, fixed_dim=0
[clip] shape=(166, 768), d=768, missing=32, malformed=0, fixed_dim=0
[clip] shape=(465, 768), d=768, missing=47, malformed=0, fixed_dim=0
Saved: C:\Users\Vaishob\boxoffice\artifacts\abl_clip_only.json

=== Run: roi_only {'u

In [14]:
# Write combined CSV
res = pd.DataFrame(rows)
res = res[["run","split","use_tab","use_text","use_clip","use_roi","MAE_lin","SMAPE","R2_log10","MAE_log10"]]
res.sort_values(["run","split"], inplace=True)
out_csv = ART_DIR / "results_ablations.csv"
res.to_csv(out_csv, index=False)
display(res.head(20))
print("Saved:", out_csv)

,run,split,use_tab,use_text,use_clip,use_roi,MAE_lin,SMAPE,R2_log10,MAE_log10
8,clip_only,test,False,False,True,False,9.240251e+07,102.210702,0.146411,0.596872
6,clip_only,train,False,False,True,False,6.036140e+07,39.408195,0.810167,0.185380
7,clip_only,val,False,False,True,False,1.038213e+08,78.306835,0.446341,0.425787
29,clip_roi,test,False,False,True,True,9.530132e+07,100.867819,0.172646,0.586152
27,clip_roi,train,False,False,True,True,5.191872e+07,36.413284,0.822214,0.171822
28,clip_roi,val,False,False,True,True,1.032518e+08,79.653982,0.435095,0.433250
11,roi_only,test,False,False,False,True,1.195794e+08,110.775959,-0.104533,0.682899
9,roi_only,train,False,False,False,True,6.875788e+07,47.175288,0.739393,0.225367
10,roi_only,val,False,False,False,True,1.450235e+08,92.001858,0.258131,0.515823
17,tab_clip,test,True,False,True,False,5.826388e+07,66.880448,0.676176,0.341584


Saved: C:\Users\Vaishob\boxoffice\artifacts\results_ablations.csv


In [18]:
# results_ablations: the combined metrics table across runs/splits
results_ablations.to_csv(ART_DIR / "results_ablations.csv", index=False)

# Save the ablation menu & run shapes (if you built a dict/list named ABLATIONS)
if 'ABLATIONS' in globals():
    save_json(ABLATIONS, ART_DIR / "ablations_menu.json")

# Optionally persist per-run metadata dicts you've built:
# save_json(run_meta, ART_DIR / f"abl_{run_name}.json") for each run_name